<a target="_blank" href="https://colab.research.google.com/github/cesarschoollectures/am-labs/blob/main/assignments/E01_Decision_Tree.ipynb">
<img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

# Aprendizado de Máquina

Nesta atividade, você irá trabalhar com o dataset Fashion MNIST utilizando modelos de classificação do sklearn.

O foco NÃO é apenas obter bons resultados, mas garantir que o experimento seja:
- correto
- reprodutível
- bem estruturado
- criticamente analisado

# Dicas importantes

## Sobre o dataset (Fashion MNIST)

- Utilize `fetch_openml` do sklearn para carregar os dados
- Use: `as_frame=False`
- Use: `mnist_784`
- Converta os rótulos para inteiro:
  
  ```python
  y = y.astype(int)
  ```

In [ ]:
## Imports

import numpy as np

from sklearn.datasets import fetch_openml
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# Questão 1

Implemente uma função load_data(seed) que:

Carregue o dataset `Fashion MNIST`
Realize a separação em treino e teste
Utilize `train_test_split` com controle de aleatoriedade
Retorne: `X_train`, `X_test`, `y_train`, `y_test`

Depois responda:
É necessário normalizar os dados para esse tipo de modelo? Justifique.

**Solução**:

In [ ]:
def load_data(seed=42):
    X, y = fetch_openml("Fashion-MNIST", version=1, as_frame=False, return_X_y=True)
    y = y.astype(int)

    X_train, X_test, y_train, y_test = train_test_split(
        X, y,
        test_size=0.2,
        random_state=seed,
        stratify=y
    )

    return X_train, X_test, y_train, y_test

### É necessário normalizar os dados para esse tipo de modelo? Justifique.

Não é estritamente necessário normalizar os dados para modelos baseados em árvores, como Random Forest e AdaBoost com estimadores do tipo árvore, porque esses algoritmos não dependem de distância entre pontos nem de escala absoluta das variáveis da mesma forma que modelos como KNN, SVM ou regressão logística. Eles tomam decisões com base em divisões por limiares nos atributos.

---



# Questão 2

Implemente as funções:

`train_random_forest(X_train, y_train, seed)`
`train_adaboost(X_train, y_train, seed)`

## Requisitos:

Utilizar os modelos do `sklearn`
Garantir reprodutibilidade com `random_state`

**Solução**:

In [ ]:
def train_random_forest(X_train, y_train, seed=42, n_estimators=100, max_depth=None):
    model = RandomForestClassifier(
        n_estimators=n_estimators,
        max_depth=max_depth,
        random_state=seed,
        n_jobs=-1
    )
    model.fit(X_train, y_train)
    return model



def train_adaboost(X_train, y_train, seed=42, n_estimators=50, max_depth=1):
    base_tree = DecisionTreeClassifier(max_depth=max_depth, random_state=seed)

    model = AdaBoostClassifier(
        estimator=base_tree,
        n_estimators=n_estimators,
        random_state=seed
    )
    model.fit(X_train, y_train)
    return model

# Questão 3

Implemente a função:

- `evaluate(model, X_test, y_test)`

Ela deve:
- Realizar predições
- Retornar a acurácia do modelo

**Solução**:

In [ ]:
def evaluate(model, X_test, y_test):
    y_pred = model.predict(X_test)
    return accuracy_score(y_test, y_pred)

**Adicione seu texto de solução aqui**.

# Questão 4

Implemente a função:

- `run_pipeline(model_type="rf", seed=42)`

Ela deve:
- Carregar os dados
- Treinar o modelo escolhido (`rf` ou `ab`)
- Avaliar o modelo
- Retornar a acurácia

**Solução**:

In [ ]:
def build_model(model_type="rf", seed=42, n_estimators=None, max_depth=None):
    if model_type == "rf":
        if n_estimators is None:
            n_estimators = 100
        return RandomForestClassifier(
            n_estimators=n_estimators,
            max_depth=max_depth,
            random_state=seed,
            n_jobs=-1
        )

    elif model_type == "ab":
        if n_estimators is None:
            n_estimators = 50
        if max_depth is None:
            max_depth = 1

        base_tree = DecisionTreeClassifier(max_depth=max_depth, random_state=seed)
        return AdaBoostClassifier(
            estimator=base_tree,
            n_estimators=n_estimators,
            random_state=seed
        )

    else:
        raise ValueError("model_type deve ser 'rf' ou 'ab'")

In [ ]:
def run_pipeline(model_type="rf", seed=42, n_estimators=None, max_depth=None, analyze_overfitting=False, return_details=False):

    X_train, X_test, y_train, y_test = load_data(seed=seed)

    model = build_model(
        model_type=model_type,
        seed=seed,
        n_estimators=n_estimators,
        max_depth=max_depth
    )

    model.fit(X_train, y_train)

    train_acc = evaluate(model, X_train, y_train)
    test_acc = evaluate(model, X_test, y_test)

    if analyze_overfitting:
        print(f"\nAcurácia treino: {train_acc:.4f}")
        print(f"Acurácia teste:  {test_acc:.4f}")

    if return_details:
        return {
            "model": model,
            "train_acc": train_acc,
            "test_acc": test_acc,
            "X_train": X_train,
            "X_test": X_test,
            "y_train": y_train,
            "y_test": y_test
        }

    return test_acc

**Em qual profundidade começa o overfitting?**

**Por que a árvore consegue 100% no treino quando max_depth=None?**

In [31]:
## Código para a análise

def analyze_overfitting_by_depth(model_type="rf", seed=42, depths=None, n_estimators=None):
    if depths is None:
        depths = [1, 10, 20, 35, 50, None]

    X_train, X_test, y_train, y_test = load_data(seed=seed)

    print(f"\n=== INICIANDO ANÁLISE DE OVERFITTING ({model_type.upper()}) ===")

    results = []

    for depth in depths:
        model = build_model(
            model_type=model_type,
            seed=seed,
            n_estimators=n_estimators,
            max_depth=depth
        )
        model.fit(X_train, y_train)

        train_acc = evaluate(model, X_train, y_train)
        test_acc = evaluate(model, X_test, y_test)

        depth_label = "None (Máx)" if depth is None else f"{depth:02d}"

        print(
            f"Profundidade: {depth_label} | "
            f"Acc Treino: {train_acc:.4f} | "
            f"Acc Teste: {test_acc:.4f}"
        )

        results.append({
            "max_depth": depth,
            "train_acc": train_acc,
            "test_acc": test_acc
        })

    print("=== FIM DA ANÁLISE ===\n")
    return results

In [ ]:
run_pipeline(model_type="rf", seed=42)
analyze_overfitting_by_depth(model_type="rf", seed=42)

run_pipeline(model_type="ab", seed=42)
analyze_overfitting_by_depth(model_type="ab", seed=42)


=== INICIANDO ANÁLISE DE OVERFITTING (RF) ===
Profundidade: 01 | Acc Treino: 0.3325 | Acc Teste: 0.3317
Profundidade: 10 | Acc Treino: 0.8835 | Acc Teste: 0.8499
Profundidade: 20 | Acc Treino: 0.9938 | Acc Teste: 0.8794
Profundidade: 35 | Acc Treino: 1.0000 | Acc Teste: 0.8807
Profundidade: 50 | Acc Treino: 1.0000 | Acc Teste: 0.8819
Profundidade: None (Máx) | Acc Treino: 1.0000 | Acc Teste: 0.8819
=== FIM DA ANÁLISE ===


=== INICIANDO ANÁLISE DE OVERFITTING (AB) ===
Profundidade: 01 | Acc Treino: 0.4951 | Acc Teste: 0.4909
Profundidade: 10 | Acc Treino: 0.9877 | Acc Teste: 0.8549
Profundidade: 20 | Acc Treino: 1.0000 | Acc Teste: 0.8696


### Em qual profundidade começa o overfitting?

Em qual profundidade começa o overfitting?

No Random Forest, os sinais de overfitting começam a aparecer a partir de max_depth = 20. Até a profundidade 10, treino e teste ainda melhoram juntos de forma mais equilibrada. Em max_depth = 20, a acurácia de treino já chega a 0.9938, enquanto a de teste fica em 0.8794. A partir daí, o treino atinge 1.0000 nas profundidades 35, 50 e None, mas o teste praticamente se estabiliza entre 0.8794 e 0.8819. Isso mostra que o modelo continua aprendendo cada vez mais o conjunto de treino, mas sem ganho real de generalização.

No AdaBoost, os sinais também começam a aparecer por volta de max_depth = 20. Em max_depth = 10, a acurácia de treino já sobe bastante para 0.9877, enquanto a de teste fica em 0.8549. Em max_depth = 20, o treino chega a 1.0000, mas o teste sobe apenas para 0.8696, ainda distante do treino. Isso indica que o modelo também começa a se ajustar excessivamente aos dados de treinamento conforme a profundidade aumenta, embora, no AdaBoost, ainda haja algum ganho no teste até esse ponto.

---
### Por que a árvore consegue 100% no treino quando max_depth=None?

Por que a árvore consegue 100% no treino quando max_depth=None?

Quando max_depth=None, a árvore pode crescer sem limite explícito, criando divisões cada vez mais específicas até praticamente memorizar os exemplos do conjunto de treino. Isso faz com que o modelo consiga separar perfeitamente os dados vistos durante o treinamento, levando a uma acurácia de 100% no treino.

No entanto, isso não significa que o modelo generaliza melhor. No experimento, isso ficou claro porque, mesmo quando o treino chegou a 1.0000, a acurácia no teste permaneceu bem menor. No Random Forest, por exemplo, com profundidades altas ou sem limite, o treino ficou em 1.0000, enquanto o teste permaneceu próximo de 0.8819. No AdaBoost, em max_depth = 20, o treino também chegou a 1.0000, mas o teste ficou em 0.8696. Isso caracteriza overfitting: o modelo aprende perfeitamente os dados de treino, mas não apresenta melhora proporcional em dados novos.

Não foi possível rodar o adaBoost até o fim (1h33min totais da execução)

---

# Questão 5

Execute o pipeline para ambos os modelos:

- Random Forest
- AdaBoost

## Apresente:
- Acurácia, Precisão, Recall e F1-Score de cada modelo

## Responda:
- Qual modelo apresentou melhor desempenho inicial?

**Solução**:

In [ ]:
def evaluate_metrics(model, X_test, y_test):
    y_pred = model.predict(X_test)

    return {
        "accuracy": accuracy_score(y_test, y_pred),
        "precision": precision_score(y_test, y_pred, average="weighted", zero_division=0),
        "recall": recall_score(y_test, y_pred, average="weighted", zero_division=0),
        "f1": f1_score(y_test, y_pred, average="weighted", zero_division=0)
    }

In [ ]:
seed = 42

# Random Forest
details_rf = run_pipeline(model_type="rf", seed=seed, return_details=True)
metrics_rf = evaluate_metrics(details_rf["model"], details_rf["X_test"], details_rf["y_test"])

print("=== Random Forest ===")
print(f"Acurácia:  {metrics_rf['accuracy']:.4f}")
print(f"Precisão:  {metrics_rf['precision']:.4f}")
print(f"Recall:    {metrics_rf['recall']:.4f}")
print(f"F1-Score:  {metrics_rf['f1']:.4f}")

# AdaBoost
details_ab = run_pipeline(model_type="ab", seed=seed, return_details=True)
metrics_ab = evaluate_metrics(details_ab["model"], details_ab["X_test"], details_ab["y_test"])

print("\n=== AdaBoost ===")
print(f"Acurácia:  {metrics_ab['accuracy']:.4f}")
print(f"Precisão:  {metrics_ab['precision']:.4f}")
print(f"Recall:    {metrics_ab['recall']:.4f}")
print(f"F1-Score:  {metrics_ab['f1']:.4f}")

=== Random Forest ===
Acurácia:  0.8819
Precisão:  0.8807
Recall:    0.8819
F1-Score:  0.8800

=== AdaBoost ===
Acurácia:  0.4909
Precisão:  0.4717
Recall:    0.4909
F1-Score:  0.4448


O modelo com melhor desempenho inicial foi o Random Forest, que apresentou acurácia de 0.8819, além de melhores valores de precisão, recall e F1-score. Já o AdaBoost, na configuração padrão utilizada, teve desempenho bem inferior, com acurácia de 0.4909, mostrando que não conseguiu capturar os padrões do Fashion MNIST com a mesma eficácia.

Isso indica que, neste experimento, o Random Forest foi muito mais adequado para o problema. A combinação de várias árvores com amostragem aleatória tornou o modelo mais robusto e capaz de generalizar melhor, enquanto o AdaBoost, com estimadores fracos rasos, mostrou desempenho limitado na configuração inicial.

---

# Questão 6

Execute o pipeline utilizando diferentes seeds (ex: 42 e 7).

## Analise:
- Os resultados mudaram?

## Responda:
- O experimento é reprodutível? Justifique.

**Solução**:

In [ ]:
seeds = [42, 7]

for seed in seeds:
    acc_rf = run_pipeline("rf", seed)
    acc_ab = run_pipeline("ab", seed)

    print(f"Seed {seed}")
    print(f"Random Forest: {acc_rf:.4f}")
    print(f"AdaBoost: {acc_ab:.4f}")
    print()

Seed 42
Random Forest: 0.8819
AdaBoost: 0.4909

Seed 7
Random Forest: 0.8801
AdaBoost: 0.4554



Os resultados mudaram ao alterar a seed, mas o comportamento dos modelos permaneceu consistente. O Random Forest variou pouco, passando de 0.8819 para 0.8801, o que sugere maior estabilidade. Já o AdaBoost apresentou variação maior, de 0.4909 para 0.4554, indicando maior sensibilidade à aleatoriedade da divisão dos dados e do treinamento.

Ainda assim, o experimento continua sendo reprodutível, pois ao fixar uma seed específica os mesmos resultados são obtidos novamente. Portanto, a reprodutibilidade não exige que seeds diferentes gerem o mesmo valor, mas sim que uma mesma seed permita repetir fielmente o experimento.

---

# Questão 7

Para pelo menos um dos modelos:

- Compare a acurácia em treino e teste

## Responda:
- Existe overfitting?
- Qual modelo tende a sofrer mais com isso?

In [ ]:
# Random Forest
details_rf = run_pipeline(model_type="rf", seed=42, return_details=True)

print("Random Forest")
print(f"Acurácia treino: {details_rf['train_acc']:.4f}")
print(f"Acurácia teste:  {details_rf['test_acc']:.4f}")

# AdaBoost
details_ab = run_pipeline(model_type="ab", seed=42, return_details=True)

print("\nAdaBoost")
print(f"Acurácia treino: {details_ab['train_acc']:.4f}")
print(f"Acurácia teste:  {details_ab['test_acc']:.4f}")

Random Forest
Acurácia treino: 1.0000
Acurácia teste:  0.8819

AdaBoost
Acurácia treino: 0.4951
Acurácia teste:  0.4909


Há forte indício de overfitting no Random Forest, pois a acurácia no treino foi 1.0, enquanto no teste ficou em 0.8819. Essa diferença mostra que o modelo se ajustou quase perfeitamente aos dados de treinamento, mas perdeu parte da capacidade de generalização em dados não vistos.

No AdaBoost, esse comportamento não apareceu da mesma forma, já que a acurácia de treino (0.4951) e a de teste (0.4909) ficaram muito próximas. Nesse caso, o problema não parece ser overfitting, mas sim baixo desempenho geral, sugerindo que o modelo, na configuração usada, ficou fraco demais para o problema.

---

# Questão 8

Varie pelo menos um hiperparâmetro em cada modelo:

- Random Forest: `n_estimators`
- AdaBoost: `n_estimators`

## Analise:
- O desempenho muda significativamente?

## Responda:
- Qual modelo é mais sensível a mudanças?

In [ ]:
seed = 42
estimators = [50, 100, 200]

print("=== Random Forest ===")
for n in estimators:
    details_rf = run_pipeline(
        model_type="rf",
        seed=seed,
        n_estimators=n,
        return_details=True
    )
    metrics_rf = evaluate_metrics(details_rf["model"], details_rf["X_test"], details_rf["y_test"])

    print(f"\nn_estimators = {n}")
    print(f"Acurácia:  {metrics_rf['accuracy']:.4f}")
    print(f"Precisão:  {metrics_rf['precision']:.4f}")
    print(f"Recall:    {metrics_rf['recall']:.4f}")
    print(f"F1-Score:  {metrics_rf['f1']:.4f}")

print("\n=== AdaBoost ===")
for n in estimators:
    details_ab = run_pipeline(
        model_type="ab",
        seed=seed,
        n_estimators=n,
        return_details=True
    )
    metrics_ab = evaluate_metrics(details_ab["model"], details_ab["X_test"], details_ab["y_test"])

    print(f"\nn_estimators = {n}")
    print(f"Acurácia:  {metrics_ab['accuracy']:.4f}")
    print(f"Precisão:  {metrics_ab['precision']:.4f}")
    print(f"Recall:    {metrics_ab['recall']:.4f}")
    print(f"F1-Score:  {metrics_ab['f1']:.4f}")

=== Random Forest ===

n_estimators = 50
Acurácia:  0.8780
Precisão:  0.8768
Recall:    0.8780
F1-Score:  0.8761

n_estimators = 100
Acurácia:  0.8819
Precisão:  0.8807
Recall:    0.8819
F1-Score:  0.8800

n_estimators = 200
Acurácia:  0.8825
Precisão:  0.8815
Recall:    0.8825
F1-Score:  0.8808

=== AdaBoost ===

n_estimators = 50
Acurácia:  0.4909
Precisão:  0.4717
Recall:    0.4909
F1-Score:  0.4448

n_estimators = 100
Acurácia:  0.5729
Precisão:  0.6115
Recall:    0.5729
F1-Score:  0.5532

n_estimators = 200
Acurácia:  0.5596
Precisão:  0.6516
Recall:    0.5596
F1-Score:  0.5424


No Random Forest, aumentar n_estimators de 50 para 100 e depois para 200 trouxe ganhos pequenos, com a acurácia passando de 0.8780 para 0.8819 e depois para 0.8825. Isso mostra que o modelo já estava relativamente estável e que adicionar mais árvores trouxe apenas melhorias marginais.

No AdaBoost, a mudança foi mais perceptível: a acurácia subiu de 0.4909 com 50 estimadores para 0.5729 com 100, mas caiu levemente para 0.5596 com 200. Isso sugere que o AdaBoost foi mais sensível a esse hiperparâmetro e que, neste experimento, aumentar indefinidamente o número de estimadores não garantiu melhora contínua.

---

# Questão 9

Responda (máx. 2 parágrafos por item):

1. A acurácia é suficiente para avaliar os modelos?
2. Como você garante que o resultado não ocorreu por acaso?
3. Cite dois possíveis problemas metodológicos neste experimento.
4. O pipeline implementado é confiável? Justifique.

1. A acurácia é suficiente para avaliar os modelos?

Não. A acurácia é importante, mas sozinha pode esconder diferenças relevantes entre os modelos, principalmente quando queremos entender a qualidade global da classificação. Métricas como precisão, recall e F1-score ajudam a observar melhor o comportamento do modelo em relação aos erros cometidos.

No caso do Fashion MNIST, como se trata de um problema multiclasse, usar múltiplas métricas permite uma análise mais completa do desempenho. Dois modelos podem ter acurácia parecida, mas se comportarem de maneira diferente na distribuição dos erros entre as classes.

---

2. Como você garante que o resultado não ocorreu por acaso?

A principal forma de reduzir a influência do acaso foi controlar a aleatoriedade com random_state, tanto na divisão dos dados quanto no treinamento dos modelos. Além disso, o experimento foi repetido com seeds diferentes, o que permitiu verificar que o Random Forest manteve desempenho muito próximo, enquanto o AdaBoost apresentou maior variação.

Isso não elimina totalmente o acaso, mas mostra que a conclusão principal não depende de uma única execução isolada: o Random Forest continuou superior ao AdaBoost nas diferentes seeds testadas.

---

3. Cite dois possíveis problemas metodológicos neste experimento.

Um possível problema metodológico é avaliar os modelos com base em apenas uma única divisão treino-teste. Isso pode fazer com que o resultado reflita particularidades daquela separação específica, e não o comportamento geral do modelo.

Outro problema é usar apenas hiperparâmetros padrão ou fazer poucas variações sem validação mais sistemática. Isso limita a análise, pois o desempenho observado pode não representar o melhor potencial de cada modelo nem permitir uma comparação totalmente justa entre eles.

---

4. O pipeline implementado é confiável? Justifique.

Sim, o pipeline implementado é confiável no sentido de organizar corretamente as etapas essenciais do experimento: carregamento dos dados, separação treino-teste, treinamento e avaliação. Além disso, a estrutura modular facilitou a comparação entre modelos, seeds e hiperparâmetros sem duplicação excessiva de código.

No entanto, essa confiabilidade é limitada ao escopo do experimento. Para uma avaliação metodologicamente mais forte, ainda seria recomendável utilizar validação cruzada, análise mais ampla de hiperparâmetros e métricas adicionais por classe.

---